<a href="https://colab.research.google.com/github/dasmiq/cs6200-retrieval-models/blob/main/retrieval-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Models and Evaluation



In this notebook, you will evaluate results ranking on a test collection. First, you'll compute the mean average precision of a baseline BM25 model. Then you'll implement a query-likelihood retrieval model based on the same inverted index.

This notebook uses the [Pyserini](http://pyserini.io/) library, a Python interface to [Anserini](http://anserini.io) and thus to [Lucene](https://lucene.apache.org/), a widely-used open-source search engine. This library is written and maintained by Jimmy Lin and his colleagues at the University of Waterloo.


We start by installing the python interface. Since it calls the underlying Lucene search engine, which is written in Java, we make sure we point to an appropriate Java installation. If like Colab you don't have Java 21, uncomment the following code and run it, or whatever makes sense for your platform.

In [1]:
## Uncomment the following code to install Java 21 on Colab
#!apt-get install openjdk-21-jre-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"
#!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
#!java -version
!$JAVA_HOME/bin/java -version

openjdk version "21.0.13.0.3" 2026-10-24
OpenJDK Runtime Environment Homebrew (build 21.0.13.0.3)
OpenJDK 64-Bit Server VM Homebrew (build 21.0.13.0.3, mixed mode, sharing)


In [2]:
!pip install pyserini
# You can change this to gpu if you have one.
# It's a pyserini dependency, but we won't need it until the next assignment.
!pip install faiss-cpu

You can use the `LuceneSearcher` to search over an index. We can initialize the searcher with a pre-built index, which Pyserini will automatically download if it hasn't already. Note that the index takes up 1.6GB of disk.

In [3]:
from pyserini.search.lucene import LuceneSearcher

searcher = LuceneSearcher.from_prebuilt_index('robust04')

W1030 14:29:22.173000 26587 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


lucene-inverted.disk45.20240803.36f7e3.tar.gz: 100%|██████████| 1.66G/1.66G [02:29<00:00, 11.9MB/s]   
Oct 30, 2026 2:32:22 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


Now we can search for a query and inspect the results:

In [4]:
hits = searcher.search('black bear attacks', 1000)

# Prints the first 10 hits
for i in range(0, 10):
    print(f'{i+1:2} {hits[i].docid:15} {hits[i].score:.5f}')

 1 LA092790-0015   7.06680
 2 LA081689-0039   6.89020
 3 FBIS4-16530     6.61630
 4 LA102589-0076   6.46450
 5 FT932-15491     6.25090
 6 FBIS3-12276     6.24630
 7 LA091090-0085   6.17030
 8 FT922-13519     6.04270
 9 LA052790-0205   5.94060
10 LA103089-0041   5.90650


The `IndexReaderUtils` class provides various methods to read the index directly. For example, we can fetch a raw document from the index given its `docid`:

In [6]:
from pyserini.index import LuceneIndexReader
from IPython.display import display, HTML

reader = LuceneIndexReader.from_prebuilt_index('robust04')

doc = reader.doc('LA092790-0015').raw()
display(HTML('<div style="font-family: Times New Roman; padding-bottom:10px">' + doc + '</div>'))

Note that the result is exactly the same as displaying the hit contents above. Given the raw text, we can obtain its analyzed form (i.e., tokenized, stemmed, stopwords removed, etc.). Here we show the first ten tokens:

In [7]:
analyzed = reader.analyze(doc)
analyzed[0:10]

['date',
 'p',
 'septemb',
 '27',
 '1990',
 'thursdai',
 'ventura',
 'counti',
 'edit',
 'p']

The index also stores the raw document vector, which we can obtain as a Python dictionary of analyzed terms to counts (i.e., term frequency).
For brevity, we only look at terms that appear more than once:

In [8]:
doc_vector = reader.get_document_vector('LA092790-0015')
{ k: v for (k, v) in doc_vector.items() if v >1 }

{'been': 11,
 'habitat': 2,
 'year': 3,
 'jenk': 4,
 'mountain': 3,
 'rees': 3,
 'would': 2,
 'debusscher': 3,
 'near': 3,
 'hungri': 3,
 'past': 2,
 'especi': 2,
 'wildlif': 2,
 'lion': 3,
 'ventura': 4,
 'anim': 9,
 'forest': 2,
 'two': 3,
 'seen': 3,
 'attack': 3,
 'black': 2,
 'i': 2,
 'food': 3,
 'counti': 4,
 'rural': 3,
 "they'r": 5,
 'sever': 3,
 'california': 2,
 'advis': 2,
 'parch': 2,
 'area': 4,
 'she': 3,
 'month': 4,
 'deer': 3,
 'famili': 2,
 'we': 2,
 'peopl': 2,
 'just': 3,
 'live': 2,
 'drought': 4,
 'ha': 3,
 'he': 2,
 'hi': 2,
 'septemb': 2,
 'yard': 2,
 'elsewher': 2,
 'natur': 2,
 'out': 4,
 'lot': 2,
 'have': 16,
 'leav': 2,
 'more': 3,
 'off': 2,
 'ojai': 3,
 'report': 4,
 'bobcat': 4,
 'coyot': 10,
 'few': 2,
 'bear': 4,
 'depart': 2,
 'author': 3,
 'dry': 2,
 'vallei': 4,
 'who': 2,
 'game': 2,
 'resid': 2,
 'hous': 3,
 'cat': 3,
 'sai': 2,
 'said': 19,
 'park': 2,
 'on': 2,
 'offici': 5,
 'wild': 3}

## Evaluating Ranked Results

We can load some standard evaluation sets such as Robust04, which contains 250 queries, or "topics" as the TREC conferences call them.

In [10]:
from pyserini.search import get_topics

"""
run this in the terminal prior to the get_topics():
curl -fL -o ~/.cache/pyserini/topics-and-qrels/topics.robust04.txt https://raw.githubusercontent.com/castorini/anserini-tools/63ceeab1dd94c1221f29b931d868e8fab67cc25c/topics-and-qrels/topics.robust04.txt
"""

topics = get_topics('robust04')
print(f'{len(topics)} queries total')

250 queries total


The topics are in a dictionary, whose keys are integers uniquely identifying each query. Each topic contains the following fields:

* `title`: TREC's term for the brief query a user might actually type;
* `description`: a longer form of the query in the form of a complete sentence; and
* `narrative`: a description of what the user is looking for and what kinds of results would be relevant or non-relevant.

In [12]:
topics[302]

{'narrative': 'Relevant documents should contain data or outbreaks of the  polio disease (large or small scale), medical protection  against the disease, reports on what has been labeled as  "post-polio" problems.  Of interest would be location of  the cases, how severe, as well as what is being done in  the "post-polio" area.',
 'description': 'Is the disease of Poliomyelitis (polio) under control in the world?',
 'title': 'Poliomyelitis and Post-Polio'}

For the purpose of your experiments, we'll divide them into a development and test set.

In [13]:
dev_topics = {k:topics[k] for k in list(topics.keys())[:125]}
test_topics = {k:topics[k] for k in list(topics.keys())[125:]}

Now, we'll fetch the relevance judgments for the Robust04 queries, which TREC calls "qrels".

In [14]:
from urllib.request import urlopen

qfile = 'https://github.com/castorini/anserini-tools/blob/63ceeab1dd94c1221f29b931d868e8fab67cc25c/topics-and-qrels/qrels.robust04.txt?raw=true'
qrels = []
for line in urlopen(qfile):
  qid, round, docid, score = line.strip().split()
  qrels.append([int(qid), 0, docid.decode('UTF-8'), int(score)])
#qrels = [line.strip().split() for line in urlopen(qfile)]

Each record in the qrel contains four fields:

1. the numeric identifier of the query;
2. the round of relevance feedback, which is here always 0;
3. the identifier of a documennt that has been judged; and
4. the relevance score of that document.

In Robust04, all relevance judgments are binary, i.e., 1 or 0. Note that not all non-relevant documents are recorded. The qrel file only contains those documents the annotators actually looked at; the vast majority of documents in the collection have not been judged. In IR evaluation, we assume that unannotated documents are non-relevant.

In [15]:
qrels[0:10]

[[301, 0, 'FBIS3-10082', 1],
 [301, 0, 'FBIS3-10169', 0],
 [301, 0, 'FBIS3-10243', 1],
 [301, 0, 'FBIS3-10319', 0],
 [301, 0, 'FBIS3-10397', 1],
 [301, 0, 'FBIS3-10491', 1],
 [301, 0, 'FBIS3-10555', 0],
 [301, 0, 'FBIS3-10622', 1],
 [301, 0, 'FBIS3-10634', 0],
 [301, 0, 'FBIS3-10635', 0]]

## Computing Mean Average Precision

The Robust04 collection uses binary relevance judgments and usually has multiple relevant results for each query. It is thus common to use **mean average precision** (MAP) to evaluate retrieval performance on it. Remember from class that MAP adds the precision at the position of each _relevant_ document in a ranked list and then divides by the total number of relevant documents. So that we don't have to scan through the entire collection, we usually evaluate MAP at some maximum rank value, such as 100 or 1000. We simply stop scanning at that maximum rank.

As we saw above, you should pass a query string (the `title` of a topic) and the desired number of results to the `search` method of the `searcher` object.

In [16]:
hits = searcher.search(dev_topics[355]['title'], 1000)
[(hit.docid, hit.score) for hit in hits[0:10]]

[('FBIS4-20436', 11.349300384521484),
 ('FBIS3-23683', 9.126500129699707),
 ('FBIS3-21238', 8.309300422668457),
 ('FBIS4-44915', 7.935699939727783),
 ('FBIS4-20602', 7.6006999015808105),
 ('FBIS4-47382', 7.529600143432617),
 ('FT943-1589', 7.480100154876709),
 ('LA071789-0059', 7.451399803161621),
 ('FBIS4-22145', 7.215099811553955),
 ('FBIS4-44667', 7.106500148773193)]

For this assignment, evaluate MAP@1000 for the list of `test_topics` we created above. You should process the `qrels` data to find the relevant results for each query.

In [36]:
## TODO [35 points]: Compute MAP@1000 for test_topics

# Relevant docs per query; judged-but-non-relevant (score 0) and unjudged docs count as non-relevant
relevant = {}
for qid, _, docid, score in qrels:
    if score > 0:
        relevant.setdefault(qid, set()).add(docid)

def average_precision(ranked_docids, rel_docs, k=1000):
    """Sum of precision at the rank of each relevant doc, divided by the total number of relevant docs."""
    if not rel_docs:
        return 0.0
    num_rel = 0
    precision_sum = 0.0
    for rank, docid in enumerate(ranked_docids[:k], start=1):
        if docid in rel_docs:
            num_rel += 1
            precision_sum += num_rel / rank
    return precision_sum / len(rel_docs)

def mean_average_precision(runs, k=1000):
    """runs: dict of qid -> ranked list of docids."""
    aps = [average_precision(runs[qid], relevant[qid], k) for qid in runs if qid in relevant]
    return sum(aps) / len(aps)

bm25_runs = {}
for qid, topic in test_topics.items():
    hits = searcher.search(topic['title'], 1000)
    bm25_runs[qid] = [hit.docid for hit in hits]

bm25_map = mean_average_precision(bm25_runs)
# print(f'BM25 MAP@1000 on {len(test_topics)} test topics: {bm25_map:.4f}')
skipped = [qid for qid in bm25_runs if qid not in relevant]
print(f'Skipped {len(skipped)} test topic(s) with no relevance judgments: {skipped}')
print(f'BM25 MAP@1000 on {len(bm25_runs) - len(skipped)} test topics: {bm25_map:.4f}')

Skipped 1 test topic(s) with no relevance judgments: [672]
BM25 MAP@1000 on 124 test topics: 0.2637


**Note on queries with no relevance judgments.** Topic 672 has no entries in `qrels`, so it has 0 relevant documents. Average precision divides by the number of relevant documents, so it is undefined for this query. Following the standard TREC convention (the one used by `trec_eval`), I skip such queries when averaging, so MAP is computed over the remaining test topics (see the printed count above).

Tradeoffs:
- **Skip (chosen):** MAP only averages queries where AP is defined, and the result is comparable to published `trec_eval` numbers. The cost is averaging over slightly fewer queries than the full test set.
- **Count as AP = 0:** keeps every query in the average, but penalizes the system for a query that no ranking could ever do well on, which lowers MAP for reasons unrelated to retrieval quality.
- The same `mean_average_precision` function is used for both BM25 and query likelihood, so both models are averaged over the same queries and the comparison stays fair.

## Implementing a Query Likelihood Model

The default `LuceneSearcher` in pyserini uses a BM25 model. In this part of the assignment, you'll interact directly with the index to implement a query likelihood retrieval model.  For more details on pyserini's index API, [see the documentation](https://github.com/castorini/pyserini/blob/master/docs/usage-indexreader.md).

We've already created an `IndexReader` for the Robust04 collection and assigned it to `reader`. Remember that we can pass documents or queries through the index's predefined analysis pipeline, a series of operations like stemming, stopword removal, etc.

In [26]:
reader.analyze('Are there black bear attacks there?')

['black', 'bear', 'attack']

For most retrieval models, including BM25 and query likelihood, we'll want to know the overall statistics for the terms we retrieve from the index. By default, pyserini will do stemming and other "analysis" steps on the terms you give it. But since we're already working with stemmed queries, we'll tell it not to perform this analysis now.

In [27]:
# Just checking that our term is already in its index form.
print(reader.analyze('bear'))

df, cf = reader.get_term_counts('bear', analyzer=None)
print(f'The document frequency is {df} and the collection frequency is {cf}.')

['bear']
The document frequency is 16429 and the collection frequency is 23545.


You might also want to know summary statistics about the collection.

In [28]:
reader.stats()

{'total_terms': 174540872,
 'documents': 528030,
 'non_empty_documents': 528030,
 'unique_terms': 923436}

Now let's traverse the postings for a term.

In [29]:
posting_list = reader.get_postings_list('bear', analyzer=None)
print(f'There are {len(posting_list)} postings.')

posting = posting_list[100]
print(f'Each posting contains a document ID ({posting.docid}), term frequency ({posting.tf}), and position list ({posting.positions}).')

print(f'searcher has a method for turning internal integer IDs like {posting.docid} into external IDs like {searcher.doc(posting.docid).docid()}.')

There are 16429 postings.
Each posting contains a document ID (3405), term frequency (1), and position list ([461]).
searcher has a method for turning internal integer IDs like 3405 into external IDs like LA031889-0009.


If, instead of mapping terms to documents, you want to use a document name to get term information, see above for the `get_document_vector` method.

Finally, it will be useful to have some information about each document in the collection, which is saved in a file in the GitHub repository.

In [30]:
import json
docfile = urlopen('https://github.com/dasmiq/cs6200-retrieval-models/raw/refs/heads/main/doclen.jsonl')
docinfo = [json.loads(line) for line in docfile]

Each record in that file contains the document `id`, its interned numeric `iid`, and the number of tokens `n`.

In [31]:
docinfo[0]

{'id': 'LA052189-0001', 'iid': 0, 'n': 6}

Use Dirichlet smoothing with $\mu$ equal to the mean document length in the collection.

In [34]:
# TODO [2 points]: Compute the mean document length from collection statistics.
mu = reader.stats()['total_terms'] / reader.stats()['documents']
print(f'mu = {mu:.2f}')

mu = 330.55


Now you should have all the information you need to implement a query likelihood retrieval model.

In [35]:
## TODO [50 points]: Implement a query likelihood model with Dirichlet smoothing.
## For a given query, index reader, and constant k, return the k documents with the highest query likelihoods.
## Think about what data structure would let you keep the top k.

import heapq
import math

collection_length = reader.stats()['total_terms']   # |C|: total word occurrences in the collection

# Cache each term's collection frequency and postings, since topics share terms
term_cache = {}

def term_stats(term, reader):
    """Return (c_qi, {internal docid: f_qi,D}) for an analyzed term."""
    if term not in term_cache:
        df, cf = reader.get_term_counts(term, analyzer=None)
        postings = reader.get_postings_list(term, analyzer=None) if cf > 0 else None
        term_cache[term] = (cf, {p.docid: p.tf for p in postings} if postings else {})
    return term_cache[term]

def query_likelihood(query, reader, k=1000):
    """Rank documents by log P(Q|D) with Dirichlet smoothing; return the top k as (docid, score)."""
    # Analyze the query once; drop terms that never occur in the collection (log 0)
    query_terms = [t for t in reader.analyze(query) if term_stats(t, reader)[0] > 0]

    # Candidate documents: those containing at least one query term
    candidates = set()
    for t in query_terms:
        candidates.update(term_stats(t, reader)[1])

    # Min-heap of size k: the root is the lowest score currently in the top k
    heap = []
    for d in candidates:
        doc_len = docinfo[d]['n']   # |D|
        score = 0.0
        for t in query_terms:
            cf, tfs = term_stats(t, reader)
            # log[(f_qi,D + mu * c_qi/|C|) / (|D| + mu)]; f_qi,D is 0 if the term is missing from D
            score += math.log((tfs.get(d, 0) + mu * cf / collection_length) / (doc_len + mu))
        if len(heap) < k:
            heapq.heappush(heap, (score, d))
        elif score > heap[0][0]:
            heapq.heapreplace(heap, (score, d))

    return [(docinfo[d]['id'], score) for score, d in sorted(heap, reverse=True)]

print(query_likelihood('black bear attacks', reader, 5))

[('LA081689-0039', -16.331461772914864), ('LA092790-0015', -16.574522427050155), ('FBIS4-16530', -16.67797004736419), ('LA102589-0076', -17.287795395086896), ('FT932-15491', -17.478998560037386)]


Now you can reuse some of your evaluation code for mean average precision from above.

In [37]:
## TODO [13 points]: Compute the MAP@1000 of your query likelihood model on the test_topics set.

ql_runs = {}
for qid, topic in test_topics.items():
    ql_runs[qid] = [docid for docid, score in query_likelihood(topic['title'], reader, 1000)]

ql_map = mean_average_precision(ql_runs)
skipped = [qid for qid in ql_runs if qid not in relevant]
print(f'Skipped {len(skipped)} test topic(s) with no relevance judgments: {skipped}')
print(f'QL (Dirichlet, mu = {mu:.2f}) MAP@1000 on {len(ql_runs) - len(skipped)} test topics: {ql_map:.4f}')
print(f'BM25 MAP@1000 on the same test topics: {bm25_map:.4f}')


Skipped 1 test topic(s) with no relevance judgments: [672]
QL (Dirichlet, mu = 330.55) MAP@1000 on 124 test topics: 0.2600
BM25 MAP@1000 on the same test topics: 0.2637
